In [ ]:
# Copyright 2026 Jair Lemmens JairLemmens@gmail.com

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

# http://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

In [ ]:
import numpy as np

In [ ]:
def ccw_angle(v,ref =np.array([1.0, 0.0])): 
    return np.arctan2(np.cross(ref, v),np.dot(ref, v)) % (2 * np.pi)

def compute_normal(ray):
    tx, ty = ray
    return np.array([-ty, tx])

def ray_intersection(p1, d1, p2, d2):
    try:
        A = np.column_stack((d1, -d2))
        b = p2 - p1
        det = np.linalg.det(A)
        t1, t2 = np.linalg.solve(A, b)
        return(p1 + t1 * d1)
    except:
        return(p1)

def joint_at_start(wall,joint):
    return(np.linalg.norm(wall.origin-joint)<1e-6)

def miter_joint(layer0,layer1):
    joint = ray_intersection(layer0.origin,layer0.axis,layer1.origin,layer1.axis)

    d1 = layer0.axis
    d2 = layer1.axis
    
    if abs(np.dot(d1,d2)) > 0.999:
        return

    p1_1 = layer0.origin + compute_normal(layer0.axis)*layer0.offset
    p2_1 =  layer1.origin + compute_normal(layer1.axis)*layer1.offset

    p1_2 = layer0.origin + compute_normal(layer0.axis)*(layer0.offset + layer0.thickness)
    p2_2  =  layer1.origin + compute_normal(layer1.axis)*(layer1.offset + layer1.thickness)

    one_start = joint_at_start(layer0,joint)
    two_start = joint_at_start(layer1,joint)

    if one_start and two_start:
        intersection0 = ray_intersection(p1_2,d1,p2_1,d2)
        intersection1 = ray_intersection(p1_1,d1,p2_2,d2)
        layer0.start_points = np.stack([intersection1,intersection0])
        layer1.start_points = np.stack([intersection0,intersection1])

    elif not one_start and not two_start:
        intersection0 = ray_intersection(p1_2,d1,p2_1,d2)
        intersection1 = ray_intersection(p1_1,d1,p2_2,d2)
        layer0.end_points = np.stack([intersection1,intersection0])
        layer1.end_points = np.stack([intersection0,intersection1])

    elif one_start and not two_start:
        intersection0 = ray_intersection(p1_1,d1,p2_1,d2)
        intersection1 = ray_intersection(p1_2,d1,p2_2,d2)
        layer0.start_points = np.stack([intersection0,intersection1])
        layer1.end_points = np.stack([intersection0,intersection1])

    elif not one_start and two_start:
        intersection0 = ray_intersection(p1_1,d1,p2_1,d2)
        intersection1 = ray_intersection(p1_2,d1,p2_2,d2)
        layer0.end_points = np.stack([intersection0,intersection1])
        layer1.start_points = np.stack([intersection0,intersection1])

def butt_joint(layer0,layer1):
        
    joint = ray_intersection(layer0.origin,layer0.axis,layer1.origin,layer1.axis)

    d1 = layer0.axis
    d2 = layer1.axis
    
    if abs(np.dot(d1,d2)) > 0.999:
        return
    p1_1 = layer0.origin + compute_normal(layer0.axis)*layer0.offset
    p2_1 =  layer1.origin + compute_normal(layer1.axis)*layer1.offset

    p1_2 = layer0.origin + compute_normal(layer0.axis)*(layer0.offset + layer0.thickness)
    p2_2  =  layer1.origin + compute_normal(layer1.axis)*(layer1.offset + layer1.thickness)

    one_start = joint_at_start(layer0,joint)
    two_start = joint_at_start(layer1,joint)

    if one_start and two_start:
        intersection0 = ray_intersection(p1_1,d1,p2_1,d2)
        intersection1 = ray_intersection(p1_2,d1,p2_1,d2)
        layer0.start_points = np.stack([intersection0,intersection1])
    
    if one_start and not two_start:
        intersection0 = ray_intersection(p1_1,d1,p2_2,d2)
        intersection1 = ray_intersection(p1_2,d1,p2_2,d2)
        layer0.start_points = np.stack([intersection0,intersection1])

    if not one_start and two_start:
        intersection0 = ray_intersection(p1_1,d1,p2_2,d2)
        intersection1 = ray_intersection(p1_2,d1,p2_2,d2)
        layer0.end_points = np.stack([intersection0,intersection1])

    if not one_start and not two_start:
        intersection0 = ray_intersection(p1_1,d1,p2_1,d2)
        intersection1 = ray_intersection(p1_2,d1,p2_1,d2)
        layer0.end_points = np.stack([intersection0,intersection1])

def solve_joint(walls):
    if len(walls)<2:
        return

    if np.linalg.norm(walls[0].start-walls[1].start)<1e-5 or np.linalg.norm(walls[0].start-walls[1].end)<1e-5:
        joint = walls[0].start
    else:
        joint = walls[0].end

    dirs = []
    flipped = []
    for wall in walls:
        if joint_at_start(wall, joint):
            v = wall.axis
            flipped.append(False)
        else:
            v = -wall.axis
            flipped.append(True)
        dirs.append(v)

    ccw = np.argsort([ccw_angle(v) for v in dirs])

    walls = [walls[i] for i in ccw]
    flipped = [flipped[i] for i in ccw]
    num_walls = len(walls)

    ###map priorities
    priorities = {}
    priorities_indices = {}
    for n,wall in enumerate(walls):
        layer_priorities = []
        for layer in wall.layers:
            layer_priorities.append(layer.priority)
            if layer.priority in priorities:
                priorities[layer.priority].append(layer)
                priorities_indices[layer.priority].append(n)
            else:
                priorities[layer.priority] = [layer]
                priorities_indices[layer.priority] = [n]

    layer_joints = {}
    for priority in reversed(sorted(priorities.keys())):
        layers = priorities[priority]
        indices = priorities_indices[priority]

        for index,layer in zip(indices,layers):
            candidate_index = index
            ###max number of connection per joint
            for n in range(4):
                
                if layer in layer_joints.values() or layer in layer_joints.keys():
                    break

                scan_cw = False
                wall_layer_index = layer.wall.layers.index(layer)
                if len(layer.wall.priorities[wall_layer_index:])>0:
                    if sorted(layer.wall.priorities[wall_layer_index:])[-1] > priority:
                        scan_cw = True

                if (flipped[index] or scan_cw) and not (flipped[index] and scan_cw):
                    #go clockwise
                    candidate_index = candidate_index - 1
                else:
                    #go counterclockwise
                    candidate_index =candidate_index + 1

                candidate_index = (candidate_index)%num_walls
                candidate_wall = walls[candidate_index]                

                flip_layers = scan_cw
                if not flipped[index] and flipped[candidate_index]:
                    flip_layers = not scan_cw
                if flipped[index] and not flipped[candidate_index]:
                    flip_layers = not scan_cw
        
                candidate_priorities = (reversed(candidate_wall.priorities) if flip_layers else candidate_wall.priorities)
                candidate_layers = (reversed(candidate_wall.layers) if flip_layers else candidate_wall.layers)

                for candidate_layer, candidate_priority in zip(candidate_layers, candidate_priorities):
                    if candidate_priority > priority:
                        #print(f'higher_priority butt joint {index} {layer.wall.layers.index(layer)},{candidate_index} {candidate_layer.wall.layers.index(candidate_layer)}')
                        butt_joint(layer,candidate_layer)
                        layer_joints[layer] = candidate_layer
                        break
                    elif candidate_priority == priority:
                        if candidate_layer in layer_joints.values() or candidate_layer in layer_joints.keys():
                            butt_joint(layer,candidate_layer)
                            layer_joints[layer] = candidate_layer
                            #print(f'visited butt joint {index} {layer.wall.layers.index(layer)},{candidate_index} {candidate_layer.wall.layers.index(candidate_layer)}')
                            break
                        else:
                            miter_joint(layer,candidate_layer)
                            layer_joints[layer] = candidate_layer
                            #print(f'miter joint {index} {layer.wall.layers.index(layer)},{candidate_index} {candidate_layer.wall.layers.index(candidate_layer)} ')
                            break
    return(layer_joints)

class Wall:
    def __init__(self,start,end,layers=None,wall_name = 'unnamed'):
        self.start = start
        self.end = end
        self.layers = [] if layers == None else layers
        self.wall_name = wall_name
    @property
    def axis(self):
        return((self.end-self.start)/np.linalg.norm(self.end-self.start))
    @property
    def origin(self):
        return(self.start)
    @property
    def priorities(self):
        return([layer.priority for layer in self.layers])
    
class WallLayer:
    def __init__(self,wall,offset,thickness,priority,layer_name='unnamed'):
        self.wall = wall
        self.offset = offset
        self.thickness = thickness
        self.priority = priority
        self.layername = layer_name
        self.start_points = np.array([self.wall.start + compute_normal(wall.axis)*self.offset,wall.start + compute_normal(wall.axis)*(self.offset+self.thickness)]) 
        self.end_points = np.array([self.wall.end + compute_normal(wall.axis)*self.offset,wall.end + compute_normal(wall.axis)*(self.offset+self.thickness)])
    @property
    def axis(self):
        return(self.wall.axis)
    @property
    def origin(self):
        return(self.wall.start)

In [ ]:

wall0 = Wall(np.array([0.0,0.0]),np.array([3.0,1.0]))
wall0.layers.append(WallLayer(wall0,0,.2,100,'concrete'))
wall0.layers.append(WallLayer(wall0,.2,.3,80,'insulation'))

wall1 = Wall(np.array([-3.0,1.0]),np.array([0.0,0.0]))
wall1.layers.append(WallLayer(wall1,0,.2,100,'concrete'))
wall1.layers.append(WallLayer(wall1,.2,.3,80,'insulation'))

wall2 = Wall(np.array([1.0,-3.0]),np.array([0.0,0.0]))
wall2.layers.append(WallLayer(wall2,0,.2,100,'concrete'))

solve_joint([wall0,wall1,wall2])